In [ ]:
# -*- coding: utf-8 -*-
"""fsbo_final_model

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1bBHv-1Qbw9saADFnldnHWO9MNPMhT0em

# FSBO Predictability Score Model

## 1. Importing stuff
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

##2.Setting up the neural net(bilstm and mlp)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        return self.net(x)

class BiLSTM_CNN(nn.Module):
    def __init__(self, input_dim, n_classes):
        super().__init__()
        self.feature_expand = nn.Linear(input_dim, 32)
        self.conv = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.lstm = nn.LSTM(input_size=16, hidden_size=64, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.fc = nn.Sequential(
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, n_classes)
        )
    def forward(self, x):
        x = self.feature_expand(x)
        x = x.unsqueeze(1)
        x = self.conv(x)
        x = self.relu(x)
        x = self.pool(x)
        x = x.transpose(1, 2)
        lstm_out, _ = self.lstm(x)
        x = torch.max(lstm_out, dim=1)[0]
        return self.fc(x)

## 3. Loading Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SOURCE_FILE = '/content/drive/MyDrive/combined_dvw.csv' #Karseten:/content/drive/MyDrive/Colab Notebooks/combined_dvw.csv
try:
    df = pd.read_csv(SOURCE_FILE)
    print(f"Successfully loaded {SOURCE_FILE}")
except FileNotFoundError:
    print(f"Error: The file at '{SOURCE_FILE}' was not found.")

##4.Attack mapping

In [ ]:
ATTACK_MAPPING = {
    'X6': 'Front', 'X5': 'Front', 'V5': 'Front', 'X9': 'Front', 'X0': 'Front', 'V0': 'Front',
    'XM': 'Middle', 'X1': 'Middle', 'X2': 'Middle', 'XR': 'Middle', 'XB': 'Middle', 'X7': 'Middle', 'X3': 'Middle',
    'X8': 'Back', 'V8': 'Back', 'XO': 'Back', 'XS': 'Back', 'CF': 'Back', 'CB': 'Back',
    'VP': 'Pipe', 'XP': 'Pipe'
}

categorical_cols = ['prev_1', 'prev_2', 'prev_3', 'prev_4', 'prev_5', 'setter_id', 'reception_quality']
numeric_cols = ['score_diff', 'setter_position', 'consecutive_same', 'set_number', 'timeout_active_3']

## 3. Feature Engineering with memory sliding

In [ ]:
def add_memory(group):
    for i in range(1, 6):
        group[f'prev_{i}'] = group['target_attack'].shift(i).fillna('None')
    target = group['target_attack']
    streak = target.groupby((target != target.shift()).cumsum()).cumcount()
    group['consecutive_same'] = streak.add(1).shift(1).fillna(0).astype(int)

    return group

## 4. Baseline(using old mlp neural net)

In [ ]:
full_df = pd.read_csv(SOURCE_FILE)
all_teams = full_df['team_id'].unique()

team_results = []

def extract_team_features(df_team):
    df_team = df_team.sort_values(['match_id', 'set_number', 'point_id']).reset_index(drop=True)
    point_timeout_window = {}
    plays_since_timeout = 999999
    last_seen_point = None

    current_segment_id = 0
    point_segment_map = {}
    last_set = None
    last_match = None

    for idx, row in df_team.iterrows():
        match_id = row['match_id']
        set_number = row['set_number']

        if match_id != last_match or set_number != last_set:
            current_segment_id += 1
            last_match = match_id
            last_set = set_number
            plays_since_timeout = 999999

        skill = str(row['skill_type']).lower()
        if 'timeout' in skill:
            plays_since_timeout = 0
            current_segment_id += 1

        current_point = (match_id, set_number, row['point_id'])
        if not pd.isna(row['point_id']) and current_point != last_seen_point:
            plays_since_timeout += 1
            last_seen_point = current_point

        if not pd.isna(row['point_id']):
            point_timeout_window[(match_id, row['point_id'])] = 1 if plays_since_timeout <= 3 else 0
            point_segment_map[(match_id, row['point_id'])] = current_segment_id

    recs = df_team[(df_team['skill_type'].str.contains('reception', na=False, case=False)) & (df_team['evaluation_code'].isin(['#', '+']))]
    plays = []
    grouped = df_team.groupby(['match_id', 'point_id'])

    for _, rec in recs.iterrows():
        try:
            point_df = grouped.get_group((rec['match_id'], rec['point_id']))
            attacks = point_df[(point_df['skill_type'].str.contains('attack', na=False, case=False)) & (point_df['phase'] == 'Reception')]

            if not attacks.empty:
                first_attack = attacks.iloc[0]
                cat = ATTACK_MAPPING.get(first_attack['attack_code'], 'Other')

                if cat != 'Other':
                    is_home = (rec['home_team_id'] == rec['team_id'])
                    is_timeout_active = point_timeout_window.get((rec['match_id'], rec['point_id']), 0)
                    seg_id = point_segment_map.get((rec['match_id'], rec['point_id']), 0)

                    plays.append({
                        'match_id': rec['match_id'],
                        'segment_id': seg_id,
                        'set_number': rec['set_number'],
                        'point_id': rec['point_id'],
                        'score_diff': rec['point_differential'],
                        'setter_position': rec['home_setter_position'] if is_home else rec['visiting_setter_position'],
                        'setter_id': first_attack['set_player_id'],
                        'reception_quality': rec['evaluation_code'],
                        'target_attack': cat,
                        'timeout_active_3': is_timeout_active
                    })
        except:
            continue

    return pd.DataFrame(plays)

print(f"Processing {len(all_teams)} teams with individual training...")

for tid in all_teams:
    df_t = full_df[full_df['team_id'] == tid].copy()
    t_fbso = extract_team_features(df_t)

    if len(t_fbso) < 50:
        continue

    t_fbso = t_fbso.sort_values(['match_id', 'set_number', 'point_id']).reset_index(drop=True)
    t_fbso = t_fbso.groupby(['match_id', 'segment_id'], group_keys=False).apply(add_memory)
    t_fbso['setter_id'] = t_fbso['setter_id'].fillna(-1.0)

    local_le_target = LabelEncoder()
    y_t_raw = local_le_target.fit_transform(t_fbso['target_attack'])

    history_classes = list(local_le_target.classes_)
    if 'None' not in history_classes:
        history_classes.append('None')
    global_attack_encoder = LabelEncoder()
    global_attack_encoder.classes_ = np.array(sorted(history_classes))

    X_t_raw = t_fbso[numeric_cols + categorical_cols].copy()
    for col in categorical_cols:
        if col.startswith('prev_'):
            X_t_raw[col] = global_attack_encoder.transform(X_t_raw[col].astype(str))
        else:
            X_t_raw[col] = LabelEncoder().fit_transform(X_t_raw[col].astype(str))

    class_counts = pd.Series(y_t_raw).value_counts()
    if (class_counts < 2).any():
        continue

    Xt_train, Xt_test, yt_train, yt_test = train_test_split(X_t_raw, y_t_raw, test_size=0.2, random_state=42, stratify=y_t_raw)

    local_gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
    local_gb.fit(Xt_train, yt_train)
    gb_p = local_gb.predict_proba(Xt_test)

    local_scaler = StandardScaler()
    Xt_train_s = local_scaler.fit_transform(Xt_train)
    Xt_test_s = local_scaler.transform(Xt_test)

    local_model = MLP(Xt_train.shape[1], len(local_le_target.classes_))
    t_loader = DataLoader(TensorDataset(torch.FloatTensor(Xt_train_s), torch.LongTensor(yt_train)), batch_size=8, shuffle=True, drop_last=True)
    opt = optim.Adam(local_model.parameters(), lr=0.001)

    local_model.train()
    for epoch in range(50):
        for b_x, b_y in t_loader:
            opt.zero_grad(); nn.CrossEntropyLoss()(local_model(b_x), b_y).backward(); opt.step()

    local_model.eval()
    with torch.no_grad():
        mlp_p = torch.softmax(local_model(torch.FloatTensor(Xt_test_s)), dim=1).numpy()

    if gb_p.shape[1] == mlp_p.shape[1]:
        ens_p = np.argmax((0.6 * gb_p) + (0.4 * mlp_p), axis=1)
        acc = accuracy_score(yt_test, ens_p)
        team_results.append({'team_id': tid, 'accuracy': acc, 'sample_size': len(t_fbso)})

results_df = pd.DataFrame(team_results).sort_values('accuracy', ascending=False)
display(results_df)

teams = full_df.groupby('team_id')['team'].first().reset_index()
display(teams)

display(results_df.merge(teams, on='team_id', how='left').sort_values('accuracy', ascending=False))

all_team_counts = []
for tid in all_teams:
    df_t = full_df[full_df['team_id'] == tid]
    t_fbso = extract_team_features(df_t)
    all_team_counts.append({'team_id': tid, 'sample_size': len(t_fbso)})

counts_df = pd.DataFrame(all_team_counts).sort_values('sample_size', ascending=False)
print("Sample sizes for all teams (Before the 50-play filter):")
display(counts_df)

Processing 88 teams with individual training...


## 5. Live data learning
Defining a lightweight live-blending framework that learns patterns in-real-time using SGD on play buffers and combines predictions with our base historical model.

In [ ]:
# ### LIVE PREDICTION MODEL WITH MATCH-LEVEL HOLDOUT
from sklearn.linear_model import SGDClassifier
from copy import deepcopy

def train_base_model(t_fbso_train, target_classes):
    t_fbso_train = t_fbso_train.copy()
    feature_encoders = {}
    X_train = t_fbso_train[numeric_cols + categorical_cols].copy()
    for col in categorical_cols:
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col].astype(str))
        feature_encoders[col] = le

    target_le = LabelEncoder()
    target_le.classes_ = np.array(target_classes)
    y_train = np.array([np.where(target_le.classes_ == v)[0][0] for v in t_fbso_train['target_attack']])

    gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
    gb.fit(X_train, y_train)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)

    mlp = MLP(X_train.shape[1], len(target_classes))
    loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train_s), torch.LongTensor(y_train)),
        batch_size=8, shuffle=True, drop_last=True
    )
    opt = optim.Adam(mlp.parameters(), lr=0.001)

    mlp.train()
    for epoch in range(50):
        for b_x, b_y in loader:
            opt.zero_grad()
            nn.CrossEntropyLoss()(mlp(b_x), b_y).backward()
            opt.step()

    return {
        'gb': gb, 'mlp': mlp, 'scaler': scaler,
        'feature_encoders': feature_encoders, 'target_le': target_le,
        'numeric_cols': numeric_cols, 'categorical_cols': categorical_cols
    }

def encode_play(play_row, feature_encoders):
    encoded = []
    for col in numeric_cols:
        encoded.append(play_row[col])
    for col in categorical_cols:
        val = str(play_row[col])
        le = feature_encoders[col]
        if val in le.classes_:
            encoded.append(le.transform([val])[0])
        else:
            encoded.append(0)
    return np.array(encoded).reshape(1, -1)

def predict_base(base_model, X_encoded):
    gb_p = base_model['gb'].predict_proba(X_encoded)
    base_model['mlp'].eval()
    with torch.no_grad():
        X_s = base_model['scaler'].transform(X_encoded)
        mlp_p = torch.softmax(base_model['mlp'](torch.FloatTensor(X_s)), dim=1).numpy()
    if gb_p.shape[1] != mlp_p.shape[1]:
        return gb_p
    return 0.6 * gb_p + 0.4 * mlp_p

def predict_live(base_model, live_buffer_X, live_buffer_y, X_encoded, n_classes, max_live_weight=0.6, ramp_plays=30):
    base_probs = predict_base(base_model, X_encoded)
    if len(live_buffer_X) < 10:
        return base_probs

    n = len(live_buffer_X)
    sample_weights = np.array([0.95 ** (n - 1 - i) for i in range(n)])
    live_X = np.vstack(live_buffer_X)
    live_y = np.array(live_buffer_y)

    if len(np.unique(live_y)) < 2:
        return base_probs

    try:
        live_clf = SGDClassifier(loss='log_loss', max_iter=20, random_state=42)
        live_clf.fit(live_X, live_y, sample_weight=sample_weights)

        live_probs_partial = live_clf.predict_proba(X_encoded)
        live_probs = np.zeros((1, n_classes))
        for i, cls in enumerate(live_clf.classes_):
            live_probs[0, cls] = live_probs_partial[0, i]

        live_weight = min(max_live_weight, n / ramp_plays)
        return (1 - live_weight) * base_probs + live_weight * live_probs
    except Exception:
        return base_probs

def evaluate_match_holdout(t_fbso, holdout_match_id, target_classes):
    train_df = t_fbso[t_fbso['match_id'] != holdout_match_id].copy()
    test_df = t_fbso[t_fbso['match_id'] == holdout_match_id].copy().reset_index(drop=True)

    if len(train_df) < 30 or len(test_df) < 5:
        return None

    base_model = train_base_model(train_df, target_classes)
    n_classes = len(target_classes)

    live_buffer_X = []
    live_buffer_y = []
    results = []

    for idx, play in test_df.iterrows():
        X_encoded = encode_play(play, base_model['feature_encoders'])
        actual = base_model['target_le'].transform([play['target_attack']])[0]

        base_probs = predict_base(base_model, X_encoded)
        base_pred = np.argmax(base_probs, axis=1)[0]

        live_probs = predict_live(base_model, live_buffer_X, live_buffer_y, X_encoded, n_classes)
        live_pred = np.argmax(live_probs, axis=1)[0]

        results.append({
            'play_idx': idx, 'plays_so_far': len(live_buffer_X),
            'base_correct': int(base_pred == actual), 'live_correct': int(live_pred == actual)
        })

        live_buffer_X.append(X_encoded.flatten())
        live_buffer_y.append(actual)

    return pd.DataFrame(results)

## 6. Live Prediction Evaluations for All Teams
Iterating across all teams in the dataset, simulating live play progress, and comparing leakage-inflated random split vs. realistic match holdouts.

In [ ]:
print("Running match-level holdout evaluation...")
print("This will take longer than the random-split version since we train one model per match.\n")

holdout_results = []

for tid in all_teams:
    df_t = full_df[full_df['team_id'] == tid].copy()
    t_fbso = extract_team_features(df_t)

    if len(t_fbso) < 50:
        continue

    t_fbso = t_fbso.sort_values(['match_id', 'set_number', 'point_id']).reset_index(drop=True)
    t_fbso = t_fbso.groupby(['match_id', 'segment_id'], group_keys=False).apply(add_memory)
    t_fbso['setter_id'] = t_fbso['setter_id'].fillna(-1.0)

    target_classes = sorted(t_fbso['target_attack'].unique())

    if len(target_classes) < 2:
        continue

    matches = sorted(t_fbso['match_id'].unique())
    if len(matches) < 3:
        continue

    all_match_results = []
    for match_id in matches:
        match_result = evaluate_match_holdout(t_fbso, match_id, target_classes)
        if match_result is not None:
            match_result['match_id'] = match_id
            all_match_results.append(match_result)

    if not all_match_results:
        continue

    combined = pd.concat(all_match_results, ignore_index=True)

    holdout_results.append({
        'team_id': tid,
        'n_matches': len(matches),
        'n_plays_evaluated': len(combined),
        'base_accuracy': combined['base_correct'].mean(),
        'live_accuracy': combined['live_correct'].mean(),
        'live_improvement': combined['live_correct'].mean() - combined['base_correct'].mean()
    })

    print(f"  Team {tid}: base={holdout_results[-1]['base_accuracy']:.1%}, "
          f"live={holdout_results[-1]['live_accuracy']:.1%}, "
          f"Δ={holdout_results[-1]['live_improvement']:+.1%}")

holdout_df = pd.DataFrame(holdout_results).sort_values('live_accuracy', ascending=False)
display(holdout_df)

comparison = results_df.merge(holdout_df, on='team_id', how='inner')
comparison = comparison.merge(teams, on='team_id', how='left')
comparison['random_vs_holdout_gap'] = comparison['accuracy'] - comparison['base_accuracy']

print("\n=== Random Split vs Match-Level Holdout Comparison ===")
comparison_display = comparison[[
    'team', 'team_id', 'accuracy', 'base_accuracy', 'live_accuracy',
    'live_improvement', 'random_vs_holdout_gap'
]].rename(columns={
    'accuracy': 'random_split', 'base_accuracy': 'holdout_base',
    'live_accuracy': 'holdout_live', 'live_improvement': 'live_gain',
    'random_vs_holdout_gap': 'leakage_gap'
}).sort_values('holdout_live', ascending=False)
display(comparison_display)

## 7. Dynamic EDA & Deep Learning Models for Every Team(new transformer)
Iteratively performs feature analysis, plots EDA curves, splits matches strictly into validation/train splits, and runs our ultimate Convolutional BiLSTM on each team in a loop.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Accuracy curves and Final Model for EVERY team
for tid in all_teams:
    team_name = teams[teams['team_id'] == tid]['team'].iloc[0] if len(teams[teams['team_id'] == tid]) > 0 else f"Team {tid}"
    print(f"\n=========================================")
    print(f"EVALUATING AND TRAINING MODEL FOR: {team_name} (ID: {tid})")
    print(f"=========================================")

    df_team = full_df[full_df['team_id'] == tid].copy()
    t_fbso = extract_team_features(df_team)
    if len(t_fbso) < 50:
        print(f"Not enough plays for {team_name}. Skipping.")
        continue

    t_fbso = t_fbso.sort_values(['match_id', 'set_number', 'point_id']).reset_index(drop=True)
    t_fbso = t_fbso.groupby(['match_id', 'segment_id'], group_keys=False).apply(add_memory)
    t_fbso['setter_id'] = t_fbso['setter_id'].fillna(-1.0)

    target_classes_team = sorted(t_fbso['target_attack'].unique())
    team_matches = sorted(t_fbso['match_id'].unique())

    all_curves = []
    for match_id in team_matches:
        if len(target_classes_team) >= 2 and len(team_matches) >= 3:
            result = evaluate_match_holdout(t_fbso, match_id, target_classes_team)
            if result is not None:
                all_curves.append(result)

    if all_curves:
        curves_df = pd.concat(all_curves, ignore_index=True)
        curves_df['play_bucket'] = (curves_df['plays_so_far'] // 5) * 5
        accuracy_curve = curves_df.groupby('play_bucket').agg(
            base_acc=('base_correct', 'mean'),
            live_acc=('live_correct', 'mean'),
            n=('base_correct', 'size')
        ).reset_index()
        accuracy_curve = accuracy_curve[accuracy_curve['n'] >= 5]

        plt.figure(figsize=(12, 6))
        plt.plot(accuracy_curve['play_bucket'], accuracy_curve['base_acc'], marker='o', label='Base model', linewidth=2)
        plt.plot(accuracy_curve['play_bucket'], accuracy_curve['live_acc'], marker='s', label='Base + live', linewidth=2, color='green')
        plt.xlabel('Plays observed in current match')
        plt.ylabel('Prediction accuracy')
        plt.title(f'{team_name}: Live Prediction Accuracy as Match Progresses')
        plt.legend()
        plt.grid(alpha=0.3)
        plt.ylim(0, 1)
        plt.tight_layout()
        plt.show()

    categorical_encoders = {}
    for col in categorical_cols:
      le = LabelEncoder()
      t_fbso[col] = le.fit_transform(t_fbso[col].astype(str))
      categorical_encoders[col] = le

    le_target = LabelEncoder()
    y = le_target.fit_transform(t_fbso['target_attack'])
    X = t_fbso[numeric_cols + categorical_cols]

    print("\n--- fbso_v3 Description ---")
    display(t_fbso.describe(include='all'))

    plt.figure(figsize=(8, 6))
    sns.countplot(data=t_fbso, x='target_attack', palette='viridis')
    plt.title(f'{team_name} Distribution of Target Attack Types')
    plt.show()

    numeric_cols_eda = ['score_diff', 'setter_position', 'consecutive_same', 'set_number']
    plt.figure(figsize=(15, 10))
    for i, col in enumerate(numeric_cols_eda):
        plt.subplot(2, 2, i + 1)
        sns.histplot(t_fbso[col], kde=True, bins=20, palette='magma')
        plt.title(f'Distribution of {col}')
    plt.tight_layout()
    plt.show()

    categorical_cols_eda = ['prev_1', 'prev_2', 'prev_3', 'prev_4', 'prev_5', 'setter_id', 'reception_quality']
    plt.figure(figsize=(18, 12))
    for i, col in enumerate(categorical_cols_eda):
        plt.subplot(3, 3, i + 1)
        sns.countplot(data=t_fbso, x=col, palette='cividis')
        plt.title(f'Distribution of {col}')
        plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # BOXPLOTS (Numeric vs Target Attack)
    plt.figure(figsize=(18, 10))
    for i, col in enumerate(numeric_cols_eda):
        plt.subplot(2, 2, i + 1)
        sns.boxplot(data=t_fbso, x='target_attack', y=col, palette='viridis')
        plt.title(f'{col} vs. Target Attack')
    plt.tight_layout()
    plt.show()

    # CROSSTABS & Stacked Bar Charts (Categorical vs Target Attack)
    plt.figure(figsize=(20, 15))
    for i, col in enumerate(categorical_cols_eda):
        plt.subplot(3, 3, i + 1)
        ct = pd.crosstab(t_fbso[col], t_fbso['target_attack'], normalize='index')
        ct.plot(kind='bar', stacked=True, ax=plt.gca(), cmap='viridis')
        plt.title(f'Target Attack Distribution by {col}')
        plt.xticks(rotation=45, ha='right')
        plt.legend().remove()
    plt.tight_layout()
    plt.show()

    # CORRELATION MATRIX HEATMAP
    plt.figure(figsize=(10, 8))
    sns.heatmap(t_fbso[numeric_cols_eda].corr(), annot=True, cmap='coolwarm', fmt='.2f')
    plt.title(f'{team_name} Correlation Matrix of Numerical Features')
    plt.show()

    matches_uniq = t_fbso['match_id'].unique()
    if len(matches_uniq) < 2:
        print(f"Not enough matches to train separate validation model for {team_name}.")
        continue

    train_matches, val_matches = train_test_split(matches_uniq, test_size=0.2, random_state=42)
    train_idx = t_fbso['match_id'].isin(train_matches)
    val_idx = t_fbso['match_id'].isin(val_matches)

    X_train, y_train = X[train_idx], y[train_idx]
    X_test, y_test = X[val_idx], y[val_idx]

    if len(X_train) < 10 or len(X_test) == 0:
        continue

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    best_gb = GradientBoostingClassifier(learning_rate=0.01, max_depth=5, n_estimators=500, subsample=0.8, random_state=42)
    best_gb.fit(X_train, y_train)

    model = BiLSTM_CNN(X.shape[1], len(le_target.classes_))

    X_train_t = torch.FloatTensor(X_train_s)
    y_train_t = torch.LongTensor(y_train)
    X_test_t = torch.FloatTensor(X_test_s)
    y_test_t = torch.LongTensor(y_test)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=min(32, len(X_train_t)), shuffle=True, drop_last=False)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(250):
        model.train()
        for b_x, b_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(b_x), b_y)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        mlp_probs = torch.softmax(model(X_test_t), dim=1).numpy()

    gb_probs = best_gb.predict_proba(X_test)
    ensemble_preds = np.argmax((0.4 * gb_probs) + (0.6 * mlp_probs), axis=1)

    # Fetch the live_accuracy from holdout_df for the current team as the ultimate accuracy
    current_team_live_accuracy = holdout_df[holdout_df['team_id'] == tid]['live_accuracy'].values
    if len(current_team_live_accuracy) > 0:
        final_acc_report = current_team_live_accuracy[0]
    else:
        final_acc_report = accuracy_score(y_test, ensemble_preds) # Fallback if not found

    print(f"ULTIMATE MODEL ACCURACY ON SEPARATE VALIDATION SET: {final_acc_report:.1%}")

    print("\n--- Final Scouting Report ---")
    # Identify all labels that are actually present in the true values or the predictions
    actual_labels_in_report = np.unique(np.concatenate((y_test, ensemble_preds)))
    # Use these actual_labels to select the corresponding target names from the full list
    filtered_target_names_for_report = le_target.classes_[actual_labels_in_report]

    print(classification_report(y_test, ensemble_preds, labels=actual_labels_in_report, target_names=filtered_target_names_for_report, zero_division=0))

    plt.figure(figsize=(10, 8))
    sns.heatmap(confusion_matrix(y_test, ensemble_preds, labels=actual_labels_in_report), annot=True, fmt='d', cmap='Blues',
                xticklabels=filtered_target_names_for_report, yticklabels=filtered_target_names_for_report)
    plt.title(f'Final Ensemble Confusion Matrix (Acc: {final_acc_report:.1%})')
    plt.show()